# Notebook 00 : exécuter les parties LLM et les essais de l'agent sur Google Colab (GPU T4)

**Projet 6, groupe 6.** Ce notebook rejoue sur un GPU gratuit ce que le projet fait avec Ollama en local : les parties LLM contre TFT (4 configurations de 10 parties de 10 manches) et les essais de l'agent (5 questions, 3 essais, 2 modèles). Mêmes versions d'Ollama et mêmes modèles que le reste du projet ; les résultats portent la mention du matériel. Sur CPU, cela prend environ une heure par modèle ; sur un GPU T4, environ cinq minutes.

Utilisation : `Exécution > Modifier le type d'exécution > T4 GPU`, puis exécuter les cellules dans l'ordre. Le dépôt est public : aucune authentification n'est nécessaire. Les résultats sont téléchargés dans une archive à la fin, à commiter à son nom dans `resultats/`.

*Ce format de notebook est la reprise des commandes exécutées à la main dans Colab le 12/11/2026 ; il n'a pas lui-même été exécuté de bout en bout.*

In [ ]:
BRANCHE = "prep/s5"      # branche du dépôt qui contient le code des expériences
!git clone -q --branch $BRANCHE https://github.com/wekt2k04/projet-groupe6-dilemme-repete-axelrod.git p6
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null 2>&1; curl -fsSL https://ollama.com/install.sh | sh > ollama_install.log 2>&1; tail -2 ollama_install.log
!pip -q install -r p6/requirements.txt 2>&1 | tail -1

In [ ]:
!nohup ollama serve > ollama.log 2>&1 &
import time; time.sleep(8)
!curl -s http://127.0.0.1:11434/api/version

In [ ]:
!ollama pull llama3.1 2>&1 | tail -1
!ollama pull qwen2.5 2>&1 | tail -1
!ollama list

In [ ]:
%%bash
cd p6
for m in llama3.1 qwen2.5; do for h in annonce non; do python scripts/run_llm_games.py --modele $m --horizon $h --parties 10 --manches 10 --adversaire TFT --materiel Colab_T4_GPU; done; done > games.log 2>&1
tail -3 games.log | cut -c1-160

In [ ]:
%%bash
cd p6
python scripts/essai_agent_ollama.py --modele llama3.1 --essais 3 > essai_llama.log 2>&1
python scripts/essai_agent_ollama.py --modele qwen2.5 --essais 3 > essai_qwen.log 2>&1
ls resultats

In [ ]:
import sys; sys.path.insert(0, "p6/src"); import glob
from experiences import charger_parties, resume_config
for f in sorted(glob.glob("p6/resultats/llm_*_TFT.jsonl")):
    meta, parties = charger_parties(f); r = resume_config(meta, parties)
    print(f"{meta['modele']:>9} annoncé={meta['horizon_annonce']!s:5} coopération {r['coop']:.0%} (IC95 {r['coop_ic95'][0]:.0%} à {r['coop_ic95'][1]:.0%}) trajectoires distinctes {r['trajectoires_distinctes']}/{r['parties']} hors format {r['defauts']}/{r['appels']}")

In [ ]:
!cd p6 && zip -q -r ../resultats_colab.zip resultats
from google.colab import files; files.download("resultats_colab.zip")